# Processing Data from Parquet Files

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly
import scipy as sp
import statsmodels as sm
import sklearn as sk
import os
import re
import time
from tqdm import tqdm

Potentially change working directory

In [ ]:
print(os.getcwd())

List files in directory

In [ ]:
print(os.listdir())

Read static reference data

In [ ]:
static_reference_data_unmerged = {}
for batch_index in [1,2]:
    static_reference_data_unmerged_filenames = os.listdir(f"palate_data_parquet/batch_{batch_index}")
    for filename in tqdm(static_reference_data_unmerged_filenames):
        if ".parquet" in filename:
            df = pd.read_parquet(f"palate_data_parquet/batch_{batch_index}/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            df.columns.name = base_name
            static_reference_data_unmerged[f"{base_name}"] = df

Read sales data for each restaurant

In [ ]:
restaurant_sales_data = {}
# Read restaurant dataframes
for batch_index in [1,2]:
    location_filenames = os.listdir(f"palate_data_parquet/batch_{batch_index}/orders_item_level")
    for location_filename in tqdm(location_filenames):
        location_id = re.sub(r'\.parquet$', '', location_filename)
        df = pd.read_parquet(f"palate_data_parquet/batch_{batch_index}/orders_item_level/" + location_filename)
        restaurant_sales_data[location_id] = df

Separate to merge

In [ ]:
tagged_data = []
static_reference_data_unmerged_without_tagged_1 = []
static_reference_data_unmerged_without_tagged_2 = []
for name, df in static_reference_data_unmerged.items():
    if "tagged" in name:
        tagged_data.append(df)
    elif "1" in name:
        df['batch'] = 1
        static_reference_data_unmerged_without_tagged_1.append(df)
    else:
        df['batch'] = 2
        static_reference_data_unmerged_without_tagged_2.append(df)

Compare batch 1's tagged v1 versus v2

In [ ]:
# Tagged items data batch 1 v1 
print(tagged_data[0].shape)
# Tagged items data batch 1 v2
print(tagged_data[1].shape)
# Tagged items data batch 2
print(tagged_data[2].shape)

In [ ]:
# Tagged items data batch 1 v1 
tagged_data_v1 = tagged_data[0].copy()
# Tagged items data batch 1 v2
tagged_data_v2 = tagged_data[1].copy()
# Tagged items data batch 2
tagged_data_batch2 = tagged_data[2].copy()
tagged_data_v2_resized = tagged_data_v2.iloc[:tagged_data[0].shape[0], :]
changes = (tagged_data_v1.map(lambda s: s.lower().replace('.', '')) != tagged_data_v2_resized.map(lambda s: s.lower().replace('.', ''))).apply(lambda r: r.any(), axis=1)
changes.sum()

In [ ]:
tagged_data_v2['item_description'] = "nan"
tagged_data_v2 = tagged_data_v2[tagged_data[2].columns.tolist()]
tagged_data_v2['batch'] = 1
tagged_data_batch2['batch'] = 2

In [ ]:
items_tagged = pd.concat([tagged_data_v2, tagged_data_batch2])
items_tagged.reset_index(drop=True, inplace=True)

In [ ]:
static_reference_data_unmerged_without_tagged = []
for df1, df2 in zip(static_reference_data_unmerged_without_tagged_1, static_reference_data_unmerged_without_tagged_2):
    print(df1.columns.name)
    print(df1.shape)
    print(df2.columns.name)
    print(df2.shape)
    static_reference_data_unmerged_without_tagged.append(pd.concat([df1, df2]))

In [ ]:
before_after_details = static_reference_data_unmerged_without_tagged[0].copy()
before_after_details.reset_index(drop=True, inplace=True)
customers = static_reference_data_unmerged_without_tagged[1].copy()
customers.reset_index(drop=True, inplace=True)
locations = static_reference_data_unmerged_without_tagged[2].copy()
locations.reset_index(drop=True, inplace=True)

In [ ]:
before_after_details['cross_over_date'] = pd.to_datetime(before_after_details['cross_over_date'])

In [ ]:
customers['age'] = customers['age'].astype('float')

In [ ]:
items_tagged['is_plant_based'] = items_tagged['is_plant_based'].str.lower().str.replace('.', '')
items_tagged['is_plant_based'].value_counts()

In [ ]:
items_tagged['brand'].value_counts()

In [ ]:
items_tagged

In [ ]:
items_tagged[items_tagged['location_id'] == 'ADPFRN3QZRCXK'].loc[18248, 'item_description']

In [ ]:
plant_based = items_tagged.loc[:,['item_name','item_description','item_type','dish_category','ingredients','is_plant_based']][items_tagged['is_plant_based'] == 'yes']

In [ ]:
plant_based['dish_category'].value_counts()

In [ ]:
pb_no_alc = plant_based[plant_based['item_type'] != 'Drink']
for row in pb_no_alc.iterrows():
    print(row)

In [ ]:
for column in locations.columns[7:-1]:
    locations[column] = locations[column].astype('float')

Readd to common list of static data

In [ ]:
static_reference_data = [before_after_details, customers, items_tagged, locations]

Store lists of dataframes

In [ ]:
%store static_reference_data
%store restaurant_sales_data